In [1]:
import sys
# !{sys.executable} -m pip install monai --quiet
# !{sys.executable} -m pip install pandas --quiet
# !{sys.executable} -m pip install wandb --quiet
# !{sys.executable} -m pip install scikit-learn --quiet
# !{sys.executable} -m pip install einops --quiet
# !{sys.executable} -m pip install torchsummary --quiet
# !{sys.executable} -m pip install timm --quiet
# !{sys.executable} -m pip install albumentations --quiet
# !{sys.executable} -m pip install matplotlib --quiet
# !{sys.executable} -m pip install nibabel --quiet
# !{sys.executable} -m pip install nnunet --no-deps --quiet
# !{sys.executable} -m pip install gdown imagecodecs  


In [ ]:
import gdown

file_id = "1Te22BVn2rO75aHr77iHUR01R65ijTdWT"
url = f"https://drive.google.com/uc?id={file_id}"
output = "masks.zip"  # change extension accordingly

gdown.download(url, output, quiet=False)

In [2]:
import time
import os
import torch
import pandas as pd
# from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
import torch.nn.functional as F

from scipy import ndimage
from glob import glob
import pandas as pd
from pathlib import Path

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

#Innat https://www.kaggle.com/code/ipythonx/cervical-spine-fracture-detection-quick-eda

from random import sample
import nibabel as nib

plt.ion()   # interactive mode

In [3]:

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
from pathlib import Path
import pandas as pd
import nibabel as nib
import numpy as np
from tifffile import imread
from tqdm import tqdm
import json
import os

# --- INPUT DATA ---
DATA_DIR = Path("data/")
CSV_PATH = DATA_DIR / "train.csv"
IMG_DIR = DATA_DIR / "train_images"
LBL_DIR =  DATA_DIR / "train_labels"  #Path('rough_masks/rough_masks')
REPO_DIR = 'nnunet'
# --- OUTPUT NNUNET DIR ---
BASE = Path("/kaggle/nnunet_raw_data_base/")
task_id = 900
task_name = "VesuviusScroll"
task_folder = f"Dataset{task_id:03d}_{task_name}"

base_dir = BASE / task_folder
imagesTr = base_dir / "imagesTr"
labelsTr = base_dir / "labelsTr"

# make folders
imagesTr.mkdir(parents=True, exist_ok=True)
labelsTr.mkdir(parents=True, exist_ok=True)


In [10]:
base_dir.mkdir(parents=True, exist_ok=True)
imagesTr.mkdir(parents=True, exist_ok=True)
labelsTr.mkdir(parents=True, exist_ok=True)
# test_dir.mkdir(parents=True, exist_ok=True)
# trained_model_dir.mkdir(parents=True, exist_ok=True)

In [11]:
from PIL import Image, ImageSequence

def safe_tiff_read(path):
    """Fast multi-page TIFF reader using PIL (handles LZW)."""
    img = Image.open(path)

    # read all pages efficiently
    frames = [np.array(frame) for frame in ImageSequence.Iterator(img)]

    # stack into (Z, H, W) or (Z, H, W, C)
    return np.stack(frames, axis=0)

In [12]:
import shutil
SPACING = [1, 1, 1]  # change if needed

df = pd.read_csv(CSV_PATH)

for _, row in tqdm(df.iterrows(), total=len(df), desc="Copying TIFFs"):
    case_id = str(row["id"])
    scroll_id = str(row["scroll_id"])

    img_path = IMG_DIR / f"{case_id}.tif"
    lbl_path = LBL_DIR / f"{case_id}.tif"

    if not img_path.exists() or not lbl_path.exists():
        print(f"Skipping missing pair: {scroll_id}")
        continue

    # destination names for nnUNet
    img_dst = imagesTr / f"{case_id}_0000.tif"
    lbl_dst = labelsTr / f"{case_id}.tif"

    json_dst_img = imagesTr / f"{case_id}.json"
    json_dst_lbl = labelsTr / f"{case_id}.json"

    # ---- COPY FILES (FAST) ----
    shutil.copy2(img_path, img_dst)
    shutil.copy2(lbl_path, lbl_dst)

    # ---- WRITE SPACING JSON ----
    spacing_info = {"spacing": SPACING}

    with open(json_dst_img, "w") as f:
        json.dump(spacing_info, f)

    with open(json_dst_lbl, "w") as f:
        json.dump(spacing_info, f)

Copying TIFFs: 100%|██████████| 806/806 [00:23<00:00, 34.69it/s]


In [5]:
import os

os.environ["nnUNet_raw"] = "/kaggle/nnunet_raw_data_base/"
os.environ["nnUNet_preprocessed"] = "/kaggle/nnunet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/nnunet_results"

In [14]:
dataset_json = {
  "name": "Vesuvius Scroll Surface Detection",
  "description": "Binary segmentation of ink regions in 3D X-ray micro-CT TIFF volumes.",
  "reference": "",
  "licence": "",
  "release": "0.1",
  "tensorImageSize": "3D",

  "channel_names": {
    "0": "CT"
  },

  "labels": {
     "background" : 0,
     "surface" : 1,
     "ignore" : 2
     
    
  },

  "file_ending": ".tif",

  "numTraining": 806,


}

with open(base_dir / "dataset.json", "w") as f:
    json.dump(dataset_json, f, indent=4)

print("✔️ dataset.json written to:", base_dir / "dataset.json")

✔️ dataset.json written to: /kaggle/nnunet_raw_data_base/Dataset900_VesuviusScroll/dataset.json


In [ ]:
os.makedirs(REPO_DIR)

In [ ]:
%cd $REPO_DIR
# # !pip install -q --upgrade pip
!git clone https://github.com/MIC-DKFZ/nnUNet.git
!{sys.executable} -m pip install   -e nnUNet/

In [6]:
sys.path.append('/kaggle/nnunet/nnUNet')

In [11]:
# %cd $REPO_DIR 
sys.path.append(REPO_DIR)
!{sys.executable} -m nnunetv2.experiment_planning.plan_and_preprocess_entrypoints -d 900 --verify_dataset_integrity -c 3d_fullres -pl nnUNetPlannerResEncM

Fingerprint extraction...
Dataset900_VesuviusScroll
Failed to open file /kaggle/nnunet_raw_data_base/Dataset900_VesuviusScroll/imagesTr/1004283650_0000.tif with reader <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'>:
Traceback (most recent call last):
  File "/kaggle/nnunet/nnUNet/nnunetv2/imageio/reader_writer_registry.py", line 49, in determine_reader_writer_from_file_ending
    _ = tmp.read_images((example_file,))
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/nnunet/nnUNet/nnunetv2/imageio/natural_image_reader_writer.py", line 43, in read_images
    assert npy_img.shape[-1] == 3 or npy_img.shape[-1] == 4, "If image has three dimensions then the last " \
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: If image has three dimensions then the last dimension must have shape 3 or 4 (RGB or RGBA). Image shape here is (320, 320, 320)
Using <class 'nnunetv2.imageio.tif_reader_writer.Tiff3DIO'> as reader/writer

##################

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="fft_conv_pytorch")

In [12]:
!{sys.executable} -m nnunetv2.run.run_training 900 3d_fullres 0  -tr nnUNetTrainerSkeletonRecall_more_DAv3 -p nnUNetResEncUNetMPlans

Using device: cuda:0
Training for 250 epochs!

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2025-11-17 11:40:32.364410: Using torch.compile...
/kaggle/nnunet/nnUNet/nnunetv2/training/nnUNetTrainer/project_specific/rsna2025/nnUNetTrainerSkeletonRecall.py:62: UserWarning: Support for ignore label with Skeleton Recall is experimental and may not work as expected
  warnings.warn(
2025-11-17 11:40:33.691688: do_dummy_2d_data_aug: False
2025-11-17 11:40:33.692882: Using splits from existing split file: /kaggle/nnunet_preprocessed/Dataset900_VesuviusScroll/splits_final.json
2025-11-17 11:40:33.693300: The split file contains 5 splits.
2025-11

In [13]:
!{sys.executable} -m nnunetv2.run.run_training 900 3d_fullres 1  -tr nnUNetTrainerSkeletonRecall_more_DAv3 -p nnUNetResEncUNetMPlans

Using device: cuda:0
Training for 250 epochs!

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2025-11-17 15:59:41.933390: Using torch.compile...
/kaggle/nnunet/nnUNet/nnunetv2/training/nnUNetTrainer/project_specific/rsna2025/nnUNetTrainerSkeletonRecall.py:62: UserWarning: Support for ignore label with Skeleton Recall is experimental and may not work as expected
  warnings.warn(
2025-11-17 15:59:43.407559: do_dummy_2d_data_aug: False
2025-11-17 15:59:43.408754: Using splits from existing split file: /kaggle/nnunet_preprocessed/Dataset900_VesuviusScroll/splits_final.json
2025-11-17 15:59:43.409139: The split file contains 5 splits.
2025-11

In [14]:
!{sys.executable} -m nnunetv2.run.run_training 900 3d_fullres 2  -tr nnUNetTrainerSkeletonRecall_more_DAv3 -p nnUNetResEncUNetMPlans

Using device: cuda:0
Training for 250 epochs!

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2025-11-17 20:22:29.973195: Using torch.compile...
/kaggle/nnunet/nnUNet/nnunetv2/training/nnUNetTrainer/project_specific/rsna2025/nnUNetTrainerSkeletonRecall.py:62: UserWarning: Support for ignore label with Skeleton Recall is experimental and may not work as expected
  warnings.warn(
2025-11-17 20:22:31.411654: do_dummy_2d_data_aug: False
2025-11-17 20:22:31.412856: Using splits from existing split file: /kaggle/nnunet_preprocessed/Dataset900_VesuviusScroll/splits_final.json
2025-11-17 20:22:31.413260: The split file contains 5 splits.
2025-11

In [ ]:
!{sys.executable} -m nnunetv2.run.run_training 900 3d_fullres 3 -tr nnUNetTrainerSkeletonRecall_more_DAv3 -p nnUNetResEncUNetMPlans

Using device: cuda:0
Training for 250 epochs!

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2025-11-18 00:28:58.064232: Using torch.compile...
/kaggle/nnunet/nnUNet/nnunetv2/training/nnUNetTrainer/project_specific/rsna2025/nnUNetTrainerSkeletonRecall.py:62: UserWarning: Support for ignore label with Skeleton Recall is experimental and may not work as expected
  warnings.warn(
2025-11-18 00:28:59.315479: do_dummy_2d_data_aug: False
2025-11-18 00:28:59.316517: Using splits from existing split file: /kaggle/nnunet_preprocessed/Dataset900_VesuviusScroll/splits_final.json
2025-11-18 00:28:59.316905: The split file contains 5 splits.
2025-11

In [22]:
!{sys.executable} -m nnunetv2.run.run_training 900 3d_fullres 4  -tr nnUNetTrainerSkeletonRecall_more_DAv3 -p nnUNetResEncUNetMPlans


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0
Training for 250 epochs!

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2025-11-17 00:48:08.357714: Using torch.compile...
################### Loading pretrained weights from file  /kaggle/nnunet_results/Dataset900_VesuviusScroll/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_4/checkpoint_best.pth ###################
Below is the list of overla

In [23]:
import shutil

shutil.make_archive("models", "zip", "/kaggle/nnunet_results")

'/kaggle/models.zip'